In [ ]:
# Se necessario, instale:
# !pip install pandas
import hmac
import os
from pathlib import Path
import hashlib
import pandas as pd
# Arquivo de entrada e saida
arquivo_entrada = Path("Pesquisa_Exame.csv")
arquivo_saida = Path("pesquisa_exame_anonimizado.csv")
# Chave secreta obrigatoria para pseudonimizacao.
os.getenv("ANON_KEY")  # Verifica se a chave esta definida
segredo = os.environ.get("ANON_KEY")
if not segredo:
    raise ValueError(
        "Defina a variavel de ambiente ANON_KEY com uma chave forte antes de executar o notebook."
    )
# Colunas que precisam ser pseudonimizadas para manter vinculacao entre registros.
colunas_para_hash = ["CPF", "RGNumero", "NomPaciente"]
# Colunas de alto risco que nao devem sair no CSV final.
colunas_para_remover = [
    "DtaNascimento",
    "NomMedico",
    "CRM",
    "CRMUF",
    "DadosClinicos",
    "LaudoMacro",
    "LaudoMicro",
    "LaudoDiagnostico",
]
def normalizar_colunas(colunas):
    return [str(col).replace("\ufeff", "").strip() for col in colunas]
def aplicar_hmac_sha256(valor, chave):
    """Aplica HMAC-SHA256 mantendo nulos e vazios como ausentes."""
    if pd.isna(valor):
        return pd.NA
    valor_str = str(valor).strip()
    if not valor_str:
        return pd.NA
    return hmac.new(
        chave.encode("utf-8"),
        valor_str.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()
try:
    if not arquivo_entrada.exists():
        raise FileNotFoundError(
            f"Arquivo de entrada nao encontrado: {arquivo_entrada.resolve()}"
        )
    # Leitura explicita para evitar erro de separador e problemas com BOM.
    df = pd.read_csv(arquivo_entrada, sep=";", encoding="utf-8-sig")
    df.columns = normalizar_colunas(df.columns)
    colunas_ausentes = [col for col in colunas_para_hash if col not in df.columns]
    if colunas_ausentes:
        raise ValueError(
            f"Colunas obrigatorias para anonimizar nao encontradas: {colunas_ausentes}"
        )
    for col in colunas_para_hash:
        df[col] = df[col].apply(lambda valor: aplicar_hmac_sha256(valor, segredo))
    colunas_presentes_para_remover = [col for col in colunas_para_remover if col in df.columns]
    if colunas_presentes_para_remover:
        df = df.drop(columns=colunas_presentes_para_remover)
    df.to_csv(arquivo_saida, index=False, encoding="utf-8-sig", sep=";")
    print(f"Arquivo salvo com sucesso: {arquivo_saida.resolve()}")
    print(f"Linhas exportadas: {len(df)}")
    print(f"Colunas removidas: {colunas_presentes_para_remover}")
    print("\nPrevia dos dados:")
    display(df.head())
except Exception as e:
    print("Erro ao anonimizar o arquivo.")
    print(f"Detalhe do erro: {e}")
    raise


Arquivo salvo com sucesso: C:\Users\rafae\OneDrive\4_Arquivos\Workspace\Python\IA\anonimizacao\pesquisa_exame_anonimizado2.csv
Linhas exportadas: 17145
Colunas removidas: ['DtaNascimento', 'NomMedico', 'CRM', 'CRMUF', 'DadosClinicos', 'LaudoMacro', 'LaudoMicro', 'LaudoDiagnostico']

Previa dos dados:


,CodRequisicao,DtaSolicitacao,DtaPrevista,Dta1aLiberacao,DtaFinalizacao,DtaColeta,NomExame,NomExameTipo,DesEvento,NomPaciente,...,ExtCID,ExtQuantidade,Campo2,Campo3,Campo4,Campo6,Campo7,Campo9,Campo10,Visualizado
0,BPS25-000001,22/09/2025 10:56,29/09/2025,25/09/2025 16:35,25/09/2025 16:35,22/09/2025,BIOPSIA PROADI-SUS,ANÁTOMO PATOLÓGICO,Laudo Concluído (Definitivo),a71f7ba02efb6a57cb14cfacee194dff45a113e269a856...,...,NaN,NaN,16187380,120812356,NaN,NaN,NaN,NaN,NaN,1
1,BPS25-000002,22/09/2025 14:00,29/09/2025,25/09/2025 18:43,25/09/2025 18:43,22/09/2025,BIOPSIA PROADI-SUS,ANÁTOMO PATOLÓGICO,Laudo Concluído (Definitivo),b7f6e3c9bce6326144b9e2fc381e46e17941a326f9d1e3...,...,NaN,NaN,16187441,120813914,NaN,NaN,NaN,NaN,NaN,1
2,BPS25-000003,22/09/2025 14:02,01/10/2025,02/10/2025 19:51,02/10/2025 19:51,22/09/2025,BIOPSIA PROADI-SUS,ANÁTOMO PATOLÓGICO,Laudo Concluído (Definitivo),99668ed0f4642bb9711684fe2eed83a79a2999977ed7d3...,...,NaN,NaN,16187436,120813985,NaN,NaN,NaN,NaN,NaN,1
3,BPS25-000003,22/09/2025 14:02,01/10/2025,02/10/2025 19:51,02/10/2025 19:51,22/09/2025,BIOPSIA PROADI-SUS,ANÁTOMO PATOLÓGICO,Laudo Concluído (Definitivo),99668ed0f4642bb9711684fe2eed83a79a2999977ed7d3...,...,NaN,NaN,16187436,120813985,NaN,NaN,NaN,NaN,NaN,1
4,BPS25-000004,22/09/2025 14:10,29/09/2025,25/09/2025 18:23,25/09/2025 18:23,22/09/2025,BIOPSIA PROADI-SUS,ANÁTOMO PATOLÓGICO,Laudo Concluído (Definitivo),089259c46225be60b120ac7fac131863a1695c3c0c2968...,...,D24,NaN,16187360,120814365,NaN,NaN,NaN,NaN,NaN,1
